## 5. GRU Baseline

In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [2]:
import json

import numpy as np

from src.dataloader import load_cgm_stats, load_tensors, load_participant_ids
from src import metrics
from src import train
from src.models.nn_baseline import GRUPredictor, HORIZONS

MODEL_NAME = 'GRU'
SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '1_baseline'
RESULTS_DIR = EXPERIMENT_DIR / 'results' / 'gru'
STABILITY_SUMMARY_PATH = RESULTS_DIR / 'gru_stability_summary.json'

In [3]:
stats = load_cgm_stats()
train_loader = load_tensors('train')  # batch_size=256 default (Training Procedure)
val_loader = load_tensors('val')
test_loader = load_tensors('test')

test_participant_ids = load_participant_ids('test')
PARTICIPANT_IDS = sorted(set(test_participant_ids))

print(f'number of train batches: ', len(train_loader), '\nnumber of valuation batches: ', len(val_loader),
      '\nnumber of test batches: ', len(test_loader))
print('number of test participants:', len(PARTICIPANT_IDS))
print('Corresponding participants ID: \n', PARTICIPANT_IDS)

sample_cgm, sample_target = next(iter(train_loader))
print('\ninput batch shape (cgm):', tuple(sample_cgm.shape))
print('target batch shape:', tuple(sample_target.shape))

number of train batches:  327 
number of valuation batches:  66 
number of test batches:  66
number of test participants: 6
Corresponding participants ID: 
 [np.str_('CGMacros-008'), np.str_('CGMacros-014'), np.str_('CGMacros-019'), np.str_('CGMacros-020'), np.str_('CGMacros-034'), np.str_('CGMacros-039')]

input batch shape (cgm): (256, 24)
target batch shape: (256, 5)


In [4]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = []  # one entry per seed, in memory -- also written to disk individually below

for seed in SEEDS:
    print(f"Test results — {MODEL_NAME} | seed={seed}\n")

    train.set_seed(seed)
    model = GRUPredictor()

    model, history = train.train_model(
        model, train_loader, val_loader, stats['cgm_mean'], stats['cgm_std'],
        checkpoint_dir=EXPERIMENT_DIR / 'checkpoints' / 'gru' /f'gru_seed{seed}',
        verbose=False,  # quiet per-epoch text logging -- the progress bar below (and the one
                         # summary line per seed, after) carry that job instead
        progress_bar=True, progress_desc=f'seed={seed}',
    )
    n_epochs = len(history['train_loss'])

    y_pred_val, y_true_val = train.predict(model, val_loader)
    y_pred_test, y_true_test = train.predict(model, test_loader)
    val_results = metrics.evaluate(y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std'])
    test_results = metrics.evaluate(y_pred_test, y_true_test, stats['cgm_mean'], stats['cgm_std'])

    test_results_per_participant = metrics.evaluate_per_participant(
        y_pred_test, y_true_test, test_participant_ids, stats['cgm_mean'], stats['cgm_std'])

    seed_result = {
        'model': MODEL_NAME, 'seed': seed, 'n_epochs_trained': n_epochs,
        'val_results': val_results, 'test_results': test_results,
        'test_results_per_participant': test_results_per_participant,
    }
    all_results.append(seed_result)

    with open(RESULTS_DIR / f'gru_seed{seed}_results.json', 'w') as f:
        json.dump(seed_result, f, indent=2)

    for h in HORIZONS:
        r = test_results[h]
        zone_a = r['ceg']['A']
        zone_ab = r['ceg']['A'] + r['ceg']['B']

        print(
            f"{h:>3} min | "
            f"RMSE {r['rmse']:.2f} | "
            f"MAE {r['mae']:.2f} | "
            f"gRMSE {r['grmse']:.2f} | "
            f"Zone A {zone_a:.2f}% | "
            f"A+B {zone_ab:.2f}%"
        )

    print(f"\n{'-' * 80}")

Test results — GRU | seed=7

device: cuda (NVIDIA A100-SXM4-40GB)  model: GRUPredictor  params: 157,957


seed=7:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 11.23 | MAE 7.59 | gRMSE 12.34 | Zone A 96.92% | A+B 99.78%
 30 min | RMSE 18.60 | MAE 12.33 | gRMSE 21.56 | Zone A 90.46% | A+B 99.54%
 60 min | RMSE 27.64 | MAE 18.20 | gRMSE 33.85 | Zone A 81.47% | A+B 98.93%
 90 min | RMSE 33.00 | MAE 22.03 | gRMSE 41.19 | Zone A 75.34% | A+B 98.30%
120 min | RMSE 36.08 | MAE 24.36 | gRMSE 45.46 | Zone A 71.11% | A+B 97.92%

--------------------------------------------------------------------------------
Test results — GRU | seed=14

device: cuda (NVIDIA A100-SXM4-40GB)  model: GRUPredictor  params: 157,957


seed=14:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 11.45 | MAE 7.75 | gRMSE 13.00 | Zone A 97.04% | A+B 99.74%
 30 min | RMSE 18.69 | MAE 12.37 | gRMSE 21.85 | Zone A 90.55% | A+B 99.48%
 60 min | RMSE 27.73 | MAE 18.23 | gRMSE 34.04 | Zone A 81.35% | A+B 98.89%
 90 min | RMSE 33.09 | MAE 21.97 | gRMSE 41.35 | Zone A 75.20% | A+B 98.30%
120 min | RMSE 36.24 | MAE 24.36 | gRMSE 45.65 | Zone A 71.04% | A+B 97.90%

--------------------------------------------------------------------------------
Test results — GRU | seed=21

device: cuda (NVIDIA A100-SXM4-40GB)  model: GRUPredictor  params: 157,957


...

 15 min | RMSE 11.39 | MAE 7.77 | gRMSE 12.78 | Zone A 97.08% | A+B 99.77%
 30 min | RMSE 18.69 | MAE 12.42 | gRMSE 21.77 | Zone A 90.37% | A+B 99.51%
 60 min | RMSE 27.80 | MAE 18.28 | gRMSE 34.04 | Zone A 81.48% | A+B 98.92%
 90 min | RMSE 33.35 | MAE 22.01 | gRMSE 41.67 | Zone A 75.38% | A+B 98.28%
120 min | RMSE 36.56 | MAE 24.38 | gRMSE 46.11 | Zone A 71.51% | A+B 97.73%

--------------------------------------------------------------------------------
Test results — GRU | seed=63

device: cuda (NVIDIA A100-SXM4-40GB)  model: GRUPredictor  params: 157,957


seed=63:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 11.42 | MAE 7.67 | gRMSE 12.83 | Zone A 97.13% | A+B 99.80%
 30 min | RMSE 18.76 | MAE 12.39 | gRMSE 21.88 | Zone A 90.58% | A+B 99.52%
 60 min | RMSE 27.63 | MAE 18.31 | gRMSE 33.66 | Zone A 81.48% | A+B 98.97%
 90 min | RMSE 32.99 | MAE 22.15 | gRMSE 41.00 | Zone A 74.87% | A+B 98.37%
120 min | RMSE 36.16 | MAE 24.55 | gRMSE 45.37 | Zone A 70.89% | A+B 98.00%

--------------------------------------------------------------------------------
Test results — GRU | seed=70

device: cuda (NVIDIA A100-SXM4-40GB)  model: GRUPredictor  params: 157,957


seed=70:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 11.30 | MAE 7.75 | gRMSE 12.49 | Zone A 97.01% | A+B 99.77%
 30 min | RMSE 18.58 | MAE 12.44 | gRMSE 21.54 | Zone A 90.35% | A+B 99.53%
 60 min | RMSE 27.49 | MAE 18.28 | gRMSE 33.52 | Zone A 81.35% | A+B 98.97%
 90 min | RMSE 32.90 | MAE 22.10 | gRMSE 40.90 | Zone A 74.91% | A+B 98.39%
120 min | RMSE 35.99 | MAE 24.62 | gRMSE 44.97 | Zone A 70.31% | A+B 98.11%

--------------------------------------------------------------------------------


In [5]:
def _extract(participant_result, metric):
    """Extract a metric from one participant's result."""
    if metric == 'zone_a':
        return participant_result['ceg']['A']
    if metric == 'zone_ab':
        return participant_result['ceg']['A'] + participant_result['ceg']['B']
    return participant_result[metric]

METRICS = ['rmse', 'mae', 'grmse', 'zone_a', 'zone_ab']

summary = {}

for h in HORIZONS:
    summary[h] = {}
    for m in METRICS:
        if m == 'zone_a':
            seed_values = [
                r['test_results'][h]['ceg']['A']
            for r in all_results
            ]
        elif m == 'zone_ab':
            seed_values = [
                r['test_results'][h]['ceg']['A']
                + r['test_results'][h]['ceg']['B']
                for r in all_results
            ]
        else:
            seed_values = [
                r['test_results'][h][m]
                for r in all_results
            ]

        mean = float(np.mean(seed_values))
        sd1 = float(np.std(seed_values, ddof=1))

        participant_means = []
        for pid in PARTICIPANT_IDS:
            values_across_seeds = [
                _extract(r['test_results_per_participant'][pid][h], m)
                for r in all_results
            ]
            participant_means.append(np.mean(values_across_seeds))
        sd2 = float(np.std(participant_means, ddof=1))
        summary[h][m] = {
            'mean': mean,
            'sd1_across_seeds': sd1,
            'sd2_across_participants': sd2,
        }

print(
    f'=== {MODEL_NAME} test set: '
    f'overall mean +/- SD_seed (SD_participant) ==='
)

for h in HORIZONS:
    s = summary[h]
    print(
        f"{h:>3}-min:  "
        f"RMSE = {s['rmse']['mean']:.2f} +/- "
        f"{s['rmse']['sd1_across_seeds']:.2f} "
        f"({s['rmse']['sd2_across_participants']:.2f}) mg/dL   "
        f"MAE = {s['mae']['mean']:.2f} +/- "
        f"{s['mae']['sd1_across_seeds']:.2f} "
        f"({s['mae']['sd2_across_participants']:.2f})   "
        f"gRMSE = {s['grmse']['mean']:.2f} +/- "
        f"{s['grmse']['sd1_across_seeds']:.2f} "
        f"({s['grmse']['sd2_across_participants']:.2f})   "
        f"Zone A = {s['zone_a']['mean']:.2f} +/- "
        f"{s['zone_a']['sd1_across_seeds']:.2f} "
        f"({s['zone_a']['sd2_across_participants']:.2f})%   "
        f"Zone A+B = {s['zone_ab']['mean']:.2f} +/- "
        f"{s['zone_ab']['sd1_across_seeds']:.2f} "
        f"({s['zone_ab']['sd2_across_participants']:.2f})%"
    )

=== GRU test set: overall mean +/- SD_seed (SD_participant) ===
 15-min:  RMSE = 11.29 +/- 0.12 (2.01) mg/dL   MAE = 7.65 +/- 0.11 (1.52)   gRMSE = 12.55 +/- 0.30 (3.22)   Zone A = 97.03 +/- 0.06 (2.13)%   Zone A+B = 99.78 +/- 0.03 (0.33)%
 30-min:  RMSE = 18.61 +/- 0.10 (3.99) mg/dL   MAE = 12.35 +/- 0.10 (2.70)   gRMSE = 21.63 +/- 0.20 (6.62)   Zone A = 90.50 +/- 0.13 (3.18)%   Zone A+B = 99.53 +/- 0.02 (0.60)%
 60-min:  RMSE = 27.59 +/- 0.13 (9.24) mg/dL   MAE = 18.25 +/- 0.12 (5.78)   gRMSE = 33.73 +/- 0.22 (14.48)   Zone A = 81.44 +/- 0.24 (5.02)%   Zone A+B = 98.94 +/- 0.03 (1.08)%
 90-min:  RMSE = 32.97 +/- 0.23 (12.97) mg/dL   MAE = 22.06 +/- 0.16 (7.98)   gRMSE = 41.07 +/- 0.37 (20.11)   Zone A = 75.13 +/- 0.34 (6.62)%   Zone A+B = 98.36 +/- 0.06 (1.64)%
120-min:  RMSE = 36.11 +/- 0.30 (15.39) mg/dL   MAE = 24.43 +/- 0.21 (9.76)   gRMSE = 45.39 +/- 0.48 (23.80)   Zone A = 70.97 +/- 0.59 (7.03)%   Zone A+B = 97.96 +/- 0.13 (2.53)%


In [ ]:
# Save one final summary JSON
with open(STABILITY_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'seeds': SEEDS,
        'n_seeds': len(SEEDS),
        'participant_ids': PARTICIPANT_IDS,
        'per_horizon_summary': summary,
    }, f, indent=2)

print(f'\nsaved per-seed results to {RESULTS_DIR}/lstm_seed*_results.json')
print(f'saved aggregate summary to {STABILITY_SUMMARY_PATH}')